# EE 587 Introduction to Robotics Homework #3

### instructor: Prof. Dr. Afşar SARANLI

### Çağdaş Güven - 2738938


1. Suppose that $a = (1,-1,2)$ and that $R = R_{x,90}$. Show by direct calculation that you have $RS(a)R^T = S(Ra)$

Rotation Matrix $R_x(90^\circ)$ and Skew Symmetric Matrix $S_a$ For $a = (1,-1,2)$ :

![alt text](question_1/q1.1.png)

Calculating $RS(a)R_T$ : Substituting R and S(a) : 

![alt text](question_1/q1.2.1.png)

![alt text](question_1/q1.2.2.png)

Ra = 

![alt text](question_1/q1.3.png)

Skew symmetric matrix of Ra :

![alt text](question_1/q1.4.png)

Then Calculation of $RS(a)R^T$ : 

![alt text](question_1/q1.5.png)



2. Given the Euler angle transformation $R = R_{z,\phi}R_{y,\theta}R_{x,\psi}$, show that $\frac{d}{dt}R = S(w) R$
Where 

$\omega = (c_{\psi}s_{\theta}\dot{\Phi}-s_{\psi}\dot{\theta})i + (s_{\psi}s_{\theta}\dot{\psi}+c_{\psi}\dot{\theta})j + (\dot{\psi}+ c_{\theta}\dot{\Phi})k$ 

(the components of i,j an k are respectively called *nutation,spin* and *precession*)

For given Euler rotation general formula first finding the rotation matrices: 

![alt text](question_2/q2.1.png)

Then we find derivation of R

Using the product rule: 

![alt text](question_2/q2.2.png)

Derivative of Each Component:

![alt text](question_2/q2.3.png)

Combining Angular Velocity Components:

![alt text](question_2/q2.4.png)

Skew-Symmetric Matrix for $w$:

![alt text](question_2/q2.5.png)

Relationship Between $\frac{dR}{dt}$ and $S(w)$:

$ \frac{dR}{dt} = S(w)R$

Where $S(w)$ encodes the contributions of $\dot{\phi}$, $\dot{\theta}$, and $\dot{\psi}$ into a single matrix.






3. Let $R_{k,\theta}$ be the rotation matrix corresponding to a rotation around an axis defined by the vector $k = (k_1,k_2,k_3)$ . Prove using the Rodrigues' Formula that we have 

$\frac{d}{d\theta}R_{k,\theta} = S(k)R_{k,\theta}$

Problem statement and Rodrigues' rotation formula:

![alt text](question_3/q3.1.png)

Derivative of $R_{k,\theta}$ with respect to $\theta$:

![alt text](question_3/q3.2.png)

Rewriting Using $R_{k,\theta}$:

![alt text](question_3/q3.3.png)

which simplifies to:

$\frac{dR_{k,\theta}}{d\theta}= S(k)R_{k,\theta}$


4. Two frames {0} and {1} are related by the Homogeneous transormation 

$$
H =
    \begin{bmatrix}
        0 & -1 & 0 & 1\\
        1 & 0 & 0 & -1\\
        0 & 0 & 1 & 0\\
        0 & 0 & 0 & 1
    \end{bmatrix}
$$

A particle has velocity $v_1(t) = (3,1,0)$ relative to frame {1}. Calculate the particle velocity relative to frame {0}?

![alt text](question_4/unnamed.png)

#### Programming:

5. In this question, you will implement a simple “Robot controller” by using the *forward kinematics* and *inverse velocity kinematics* of a planar RR manipulator. We know all the kinematics and velocity kinematics equations of this planar robot from lecture notes. You can use either Matlab or Python to implement this question

**a.**  Write code to “simulate” a Planar RR robot with link lengths 𝑙1 =3 and 𝑙2 =
2 units. Define the angular ranges for the joints as 𝜃1 ={−90°,90°} and 𝜃2 =
{−170°,170°}. For any given pair of joint variables, you should be able to 
plot the robot on a plane (different colors for each link? Small circles for the 
joints? A fat point for the end effector?) 

**b.** Define two points in the robot workspace: The start point $p_s$ and goal point $p_g$ in the robot work-space. (You can later experiment with different start and end points) 

**c.** We would like to implement a discrete-time proportional position controller in the work-space that would take the robot from the initial point to the goal point. For this, you will periodically (at each time step): (1) calculate the position error, (2) define a required end-effector velocity vector (your controller), (3) transform this into required joint velocities, (4) execute the joint motion to determine the new end-effector position, (5) repeat the cycle.

**d.** Experiment with different sampling periods and controller Proportional Gains. 

**e.** Present your results in the report as “time snapshots” of the robot configuration and/or end effector positions. Also see (g) below. 

**f.** You should be able to detect and raise an alarm if a singularity is encountered during the control of the robot. 

**g.** Have a video recording (screen capture) of the execution of your controller for some example cases. Upload to YouTube as “unlisted” videos and share the link in your homework report. 

In [ ]:
import numpy as np
import scipy.linalg
from scipy.linalg import expm
import roboticstoolbox as rtb
from roboticstoolbox import *
from roboticstoolbox import ERobot2, Link2
from spatialmath import *
from spatialmath.base import *
from spatialgeometry import Sphere, Box
from swift import Swift
from spatialmath import SE3, SE2
from roboticstoolbox import quintic, trapezoidal, mtraj, mstraj, xplot, ctraj
from mpl_toolkits.mplot3d import Axes3D
from roboticstoolbox.backends.PyPlot import PyPlot2, RobotPlot2
import math
import time
from math import pi
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.widgets import Slider
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle, Circle
np.set_printoptions(linewidth=100, formatter={'float': lambda x: f"{x:8.4g}" if abs(x) > 1e-10 else f"{0:8.4g}"})

%matplotlib widget

In [ ]:
# Define link lengths
l1 = 3  # Length of link1
l2 = 2  # Length of link2

# Robot simulation function
def plot_robot(q, l1, l2, ax):
    """
    Plot the planar RR robot in a given configuration.
    """
    # Forward kinematics
    theta1, theta2 = q
    x1, y1 = l1 * np.cos(theta1), l1 * np.sin(theta1)  # End of link1
    x2, y2 = x1 + l2 * np.cos(theta1 + theta2), y1 + l2 * np.sin(theta1 + theta2)  # End of link2

    # Clear the axis
    ax.clear()
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_aspect('equal')
    ax.grid()

    # Plot link1 (blue rectangle)
    ax.add_patch(Rectangle((0, 0), l1, 0.2, angle=np.degrees(theta1), color='blue', alpha=0.7))
    # Plot link2 (red rectangle)
    ax.add_patch(Rectangle((x1, y1), l2, 0.2, angle=np.degrees(theta1 + theta2), color='red', alpha=0.7))

    # Plot joints and end-effector
    ax.plot([0, x1, x2], [0, y1, y2], '-o', markersize=8, color='black')
    ax.add_patch(Circle((x2, y2), radius=0.1, color='orange'))  # End-effector
    ax.set_title("Planar RR Robot")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")

# Test the robot plot
fig, ax = plt.subplots()
q_test = [np.deg2rad(45), np.deg2rad(-45)]  # Joint angles in radians
plot_robot(q_test, l1, l2, ax)
plt.show()


In [ ]:
# Start and goal points in the workspace
p_s = np.array([3, 2])  # Start point
p_g = np.array([4, -1])  # Goal point

# Print workspace points
print("Start Point (p_s):", p_s)
print("Goal Point (p_g):", p_g)

In [ ]:
def proportional_controller(l1, l2, p_s, p_g, Kp=1.0, dt=0.1, max_steps=100):
    """
    Proportional position controller for a planar RR robot.
    """
    def inverse_kinematics(x, y, l1, l2):
        c2 = (x**2 + y**2 - l1**2 - l2**2) / (2 * l1 * l2)
        if abs(c2) > 1:
            raise ValueError("Target point is outside the workspace!")
        s2 = np.sqrt(1 - c2**2)
        theta2 = np.arctan2(s2, c2)
        theta1 = np.arctan2(y, x) - np.arctan2(l2 * s2, l1 + l2 * c2)
        return [theta1, theta2]

    # Initial configuration
    q = inverse_kinematics(p_s[0], p_s[1], l1, l2)

    # Store trajectory
    trajectory = [list(q)]  # Ensure the trajectory stores lists

    for step in range(max_steps):
        # Forward kinematics to compute current end-effector position
        theta1, theta2 = q
        x_current = l1 * np.cos(theta1) + l2 * np.cos(theta1 + theta2)
        y_current = l1 * np.sin(theta1) + l2 * np.sin(theta1 + theta2)
        p_current = np.array([x_current, y_current])

        # Calculate position error
        error = p_g - p_current

        # Check for convergence
        if np.linalg.norm(error) < 1e-2:
            print(f"Goal reached in {step} steps!")
            break

        # Compute end-effector velocity
        v = Kp * error

        # Compute Jacobian
        J = np.array([
            [-l1 * np.sin(theta1) - l2 * np.sin(theta1 + theta2), -l2 * np.sin(theta1 + theta2)],
            [l1 * np.cos(theta1) + l2 * np.cos(theta1 + theta2),  l2 * np.cos(theta1 + theta2)]
        ])
        J_inv = np.linalg.pinv(J)  # Pseudo-inverse for safety

        # Compute joint velocities
        q_dot = J_inv @ v

        # Update joint positions
        q = q + q_dot * dt
        trajectory.append(list(q))  # Append the current joint configuration as a list

    return trajectory






In [ ]:
trajectory = proportional_controller(l1, l2, p_s, p_g, Kp=0.5, dt=0.1, max_steps=200)
# Initialize the figure and axes
fig, ax = plt.subplots()
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_aspect('equal')
ax.grid()

ax.plot(p_s[0], p_s[1], 'go', label='Start Point')  # Start point (green circle)
ax.plot(p_g[0], p_g[1], 'ro', label='Goal Point')  # Goal point (red circle)

# Add elements for the links and the end-effector
link1_line, = ax.plot([], [], '-o', color='blue', label='Link 1')  # Link 1
link2_line, = ax.plot([], [], '-o', color='red', label='Link 2')   # Link 2
end_effector_circle = ax.plot([], [], 'o', color='orange', label='End-Effector')[0]  # End-effector

trace_x, trace_y = [], []  # To store the end-effector path
trace, = ax.plot([], [], '--', color='orange', label='End-Effector Path') # path for end effector

def is_singular(J):
    """
    Detect singular configurations based on Jacobian condition number.
    """
    cond = np.linalg.cond(J)
    return cond > 1e3  # Example threshold for singularity

# Animation update function
def update_plot(frame):
    """
    Update the plot for the current frame of the animation.
    """
    # Extract joint angles for the current frame
    q = trajectory[frame]  # Get joint angles for the current frame
    theta1, theta2 = q

    # Forward kinematics
    x1, y1 = l1 * np.cos(theta1), l1 * np.sin(theta1)  # End of link1
    x2, y2 = x1 + l2 * np.cos(theta1 + theta2), y1 + l2 * np.sin(theta1 + theta2)  # End of link2

    # Compute the Jacobian
    J = np.array([
        [-l1 * np.sin(theta1) - l2 * np.sin(theta1 + theta2), -l2 * np.sin(theta1 + theta2)],
        [l1 * np.cos(theta1) + l2 * np.cos(theta1 + theta2),  l2 * np.cos(theta1 + theta2)]
    ])
    
    # Check for singularity
    if is_singular(J):
        end_effector_circle.set_color('purple')  # Highlight singularities with a purple end-effector
    else:
        end_effector_circle.set_color('orange')  # Reset to orange

    # Update link1
    link1_line.set_data([0, x1], [0, y1])  # Base to end of link1

    # Update link2
    link2_line.set_data([x1, x2], [y1, y2])  # End of link1 to end of link2

    # Update end-effector
    end_effector_circle.set_data([x2], [y2])  # Position of end-effector

    # Update the trace
    trace_x.append(x2)
    trace_y.append(y2)
    trace.set_data(trace_x, trace_y)

# Create the animation
ani = FuncAnimation(fig, update_plot, frames=len(trajectory), interval=100, repeat=True)

plt.legend()
plt.show()

In [ ]:
# Create sliders for Kp and dt
ax_kp = plt.axes([0.25, 0.01, 0.65, 0.03])  # Position of Kp slider
ax_dt = plt.axes([0.25, 0.05, 0.65, 0.03])  # Position of dt slider

slider_kp = Slider(ax_kp, 'Kp', 0.1, 2.0, valinit=0.5)
slider_dt = Slider(ax_dt, 'dt', 0.01, 0.2, valinit=0.1)

def update_controller(val):
    """
    Update the controller with new Kp and dt values from sliders.
    """
    global trajectory
    Kp = slider_kp.val
    dt = slider_dt.val
    trajectory = proportional_controller(l1, l2, p_s, p_g, Kp=Kp, dt=dt, max_steps=200)
    ani.event_source.stop()
    ani.event_source.start()

slider_kp.on_changed(update_controller)
slider_dt.on_changed(update_controller)

In [ ]:
# Convert trajectory to radians for plotting
trajectory = np.array(trajectory)

# Plot joint trajectories
plt.figure()
for i in range(trajectory.shape[1]):
    plt.plot(trajectory[:, i], label=f'Joint {i + 1}')
plt.legend()
plt.xlabel('Point Index')
plt.ylabel('Joint Angle (rad)')
plt.title('Joint Trajectories')
plt.grid()
plt.show()

6. Review the P. Corke book and *Robotics Toolbox for Python* documentation to list (with short explanations) fundamental functions that calculate the *forward* and *inverse velocity kinematics* of robot arms as well as the *inverse position kinematics*. You may illustrate some examples that use these functions.

Programming Tasks
Planar RR Manipulator Simulation (Question 5)

Steps:

    Robot Simulation (Part a):
        Create a visualization for the planar RR robot in Python/Matlab.
        Use link lengths l1=3l1​=3 and l2=2l2​=2, plotting the links and joints for a given (θ1,θ2)(θ1​,θ2​).

    Workspace Definition (Part b):
        Define start psps​ and goal pgpg​ positions in Cartesian coordinates within the workspace.

    Discrete-Time Controller (Part c):
        Implement a proportional controller:
            Compute the position error e=pg−pcurrente=pg​−pcurrent​.
            Calculate end-effector velocity: vee=Kp⋅evee​=Kp​⋅e (adjust KpKp​ experimentally).
            Use the Jacobian to compute joint velocities: q˙=J−1⋅veeq˙​=J−1⋅vee​.
            Update joint positions q=q+q˙⋅dtq=q+q˙​⋅dt.
        Plot the robot configuration and end-effector trajectory.

    Experimentation (Parts d-g):
        Vary the proportional gain KpKp​ and time step dtdt.
        Visualize snapshots of the robot moving toward the goal.
        Handle singularity detection by checking the determinant of the Jacobian matrix.

    Optional:
        Record and upload a video of the controller executing for different cases.

Functions in P. Corke's Toolbox (Question 6)

    Explore Python’s Robotics Toolbox or Matlab equivalent.
    Identify and list functions for:
        Forward kinematics.
        Inverse kinematics.
        Velocity transformations.
    Provide short explanations and include examples.